# LeCun et al. 1990, "Handwritten Digit Recognition: Applications of Neural Net Chips and Automatic Learning"
> The following tries to reproduce the original paper. Note that the digits dataset actually used in the paper could not be found and [MNIST 784](https://www.openml.org/search?type=data&status=active&id=554) is used instead

## References

* LeCun et al. 1990, _Handwritten Digit Recognition: Applications of Neural Net Chips and Automatic Learning_, [Neurocomputing](https://link.springer.com/chapter/10.1007/978-3-642-76153-9_35)

specifics in LeCun et al. 1990:

* neural net
    * weight initialization: uniformly at random $\in [-2.4 / F_i, 2.4 / F_i]$ with $F_i = $ number of inputs of the unit
    * "tanh activation": $A \cdot \tanh (S \cdot a)$ with $A = 1.716$, $S = 2/3$ and $a = \text{weights} \cdot \text{input}$
    * 256 input (16 x 16 pixel images)
    * layer #1: 
        * convolution with 12 5x5-kernels and stride 2 (output: 8 x 8 x 12 = 786 "units")
        * tanh activation
        * $F_i = 5 \cdot 5 \cdot n_\text{input-channels} = 5 \cdot 5 \cdot 1 = 25$
    * layer #2: 
        * convolution with 12 5x5-kernels and stride 2 (output: 4 x 4 x 12 = 192 "units")
        * tanh activation
        * $F_i = 5 \cdot 5 \cdot n_\text{input-channels} = 5 \cdot 5 \cdot 12 = 300$
    * layer #3:
        * dense with 30 neurons
        * tanh activation
        * $F_i = 4 \cdot 4 \cdot 12 = 192$
    * layer #4:
        * dense output layer with 10 neurons
        * tanh activation
        * $F_i = 30$
* target: vector of 10 values either 1 or -1 (so 9x -1 and 1x 1)
* loss: mean squared error between prediction and target (paper reached 1.8e-2 on test and 2.5e-3 on train)
* error rates: 0.14% on train, 5% on test
* training:
    * stochastic gradient descent (1 sample per backpropagation)
    * samples always in the same order, no shuffling
    * 23 or 30 epochs, paper is ambiguous
    * learning rate was set using some not defined 2nd order derivative method

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns
import torch
import torch.optim as optim
from sklearn import metrics
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from torch.optim import SGD
from torch.utils.data import DataLoader

import random_neural_net_models.convolution_lecun1990 as conv_lecun1990
import random_neural_net_models.data as rnnm_data
import random_neural_net_models.learner as rnnm_learner
import random_neural_net_models.losses as rnnm_losses
import random_neural_net_models.utils as utils

sns.set_theme()

In [ ]:
DO_OVERFITTING_ONLY = False

In [ ]:
mnist = fetch_openml("mnist_784", version=1, cache=True, parser="auto")

Setting seeds

In [ ]:
utils.make_deterministic(42)

Getting device

In [ ]:
device = utils.get_device()
device

In [ ]:
X = pl.from_pandas(mnist["data"])
y = pl.from_pandas(mnist["target"])
X.shape, y.shape

Selecting a few images to overfit on

In [ ]:
Xy = pl.concat([X, y.to_frame()], how="horizontal")
Xy.head(3)

In [ ]:
feature_cols = X.columns
y_col = "class"
y_col, feature_cols[:3]

In [ ]:
n = 100
label = "5"

X0_overfit = Xy.filter(pl.col(y_col).eq(label)).head(n).drop(y_col)
y0_overfit = Xy.filter(pl.col(y_col).eq(label)).head(n)[y_col]
X0_overfit.head(), y0_overfit.head()

In [ ]:
X0_overfit = X0_overfit.to_pandas()
y0_overfit = y0_overfit.to_pandas()
X = X.to_pandas()
y = y.to_pandas()

## Defining dataset and dataloader

In [ ]:
ds = conv_lecun1990.DigitsDataset(X0_overfit, y0_overfit)

In [ ]:
item = ds[4]
plt.imshow(item[0], cmap="gray", origin="upper")
plt.title(f"Label: {item[1]}")
plt.axis("off")
plt.tight_layout()

defining a dataloader

In [ ]:
batch_size = 1
dataloader = DataLoader(ds, batch_size=batch_size, shuffle=False)

In [ ]:
train_features, train_labels = next(iter(dataloader))

inspecting one image of the first batch

In [ ]:
print(f"Feature batch shape: {train_features.size()}")
print(f"Labels batch shape: {train_labels.size()}")
img = train_features[0]
label = train_labels[0]
plt.imshow(img, cmap="gray")
plt.axis("off")
plt.show()
print(f"Label: {label}")

estimating convolution block height / width

In [ ]:
(
    conv_lecun1990.calc_conv_output_dim(28, 5, 2, 2),
    conv_lecun1990.calc_conv_output_dim(14, 5, 2, 2),
)

## Custom 2d convolution

defining a 2d convolution component

In [ ]:
kh = kw = 5
n_in_channels = 1
n_out_channels = 1

myconv2d = conv_lecun1990.Conv2d(
    edge=28,
    n_in_channels=n_in_channels,
    n_out_channels=n_out_channels,
    kernel_width=kw,
    kernel_height=kh,
    stride=2,
    padding=2,
    dilation=1,
)

In [ ]:
train_features, train_labels = next(iter(dataloader))
train_features = train_features.unsqueeze(dim=1)

print(f"{train_features.shape=}")

applying the convolution to an image

In [ ]:
conv_features = myconv2d(train_features)

print(f"{conv_features.shape=}")

visualizing the effect

In [ ]:
label = train_labels[0]
fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(5, 5))

ax = axs[0]
img = train_features[0][0]
ax.imshow(img, cmap="gray")
ax.axis("off")

ax = axs[1]
img = conv_features.detach().numpy()[0][0]
ax.imshow(img, cmap="gray")
ax.axis("off")

plt.show()
print(f"Label: {label}")

## creating dataloaders

data for overfitting sanity check

In [ ]:
ds_train_overfit = rnnm_data.MNISTDatasetWithLabels(X0_overfit, y0_overfit)

In [ ]:
dl_train_overfit = DataLoader(
    ds_train_overfit,
    batch_size=1,
    collate_fn=rnnm_data.collate_mnist_dataset_to_block_with_labels,
    shuffle=True,
)

data for actual training and validation

In [ ]:
n = 10_000  # bit larger than the number of samples in the paper
X0, X2, y0, y2 = train_test_split(
    X.iloc[:n], y.iloc[:n], test_size=0.2, random_state=42
)
X0, X1, y0, y1 = train_test_split(X0, y0, test_size=0.2, random_state=42)
len(X0), len(X1), len(X2)

In [ ]:
ds_train = rnnm_data.MNISTDatasetWithLabels(X0, y0)
ds_valid = rnnm_data.MNISTDatasetWithLabels(X1, y1)
ds_test = rnnm_data.MNISTDatasetWithLabels(X2, y2)

In [ ]:
ds_train[0];

In [ ]:
dl_train = DataLoader(
    ds_train,
    batch_size=1,
    collate_fn=rnnm_data.collate_mnist_dataset_to_block_with_labels,
    shuffle=True,
)
dl_valid = DataLoader(
    ds_valid,
    batch_size=200,
    collate_fn=rnnm_data.collate_mnist_dataset_to_block_with_labels,
    shuffle=False,
)
dl_test = DataLoader(
    ds_test,
    batch_size=200,
    collate_fn=rnnm_data.collate_mnist_dataset_to_block_with_labels,
    shuffle=False,
)

## trying to overfit

using `densify_y` to convert a label to a vector of -1/1, i.e. False/True

In [ ]:
dtype = torch.float32

In [ ]:
model = conv_lecun1990.Model2(lecun_init=True, lecun_act=True, dtype=dtype)

In [ ]:
n_epochs = 10
lr = 1e-2
optimizer = optim.SGD(model.parameters(), lr=lr)  # , momentum=1e-3
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer=optimizer,
    max_lr=lr,
    epochs=n_epochs,
    steps_per_epoch=len(dl_train),
)
loss = rnnm_losses.MSELossMNIST1HotLabel()
save_dir = Path("../../models").absolute().resolve()
save_dir.mkdir(exist_ok=True)

callbacks = rnnm_learner.get_default_train_callbacks(
    activation_callback_args=rnnm_learner.TelemetryArgs(100, 4, (r".*act.*",)),
    gradient_callback_args=rnnm_learner.TelemetryArgs(100, 4, (r"conv\d$", r"lin\d")),
    parameter_callback_args=rnnm_learner.TelemetryArgs(100, 4, (r"conv\d$", r"lin\d")),
    scheduler=scheduler,
)


lr_find_callback = rnnm_learner.LRFinderCallback(1e-5, 10, 100)

learner = rnnm_learner.Learner(
    model,
    optimizer,
    loss,
    callbacks=callbacks,
    save_dir=save_dir,
    device=device,
)

In [ ]:
learner.find_learning_rate(
    dl_train_overfit, n_epochs=2, lr_find_callback=lr_find_callback
)

In [ ]:
lr_find_callback.plot()

In [ ]:
learner.fit(dl_train_overfit, n_epochs=n_epochs)

In [ ]:
rnnm_learner.plot(callbacks)

In [ ]:
optimizer = optim.SGD(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer=optimizer,
    max_lr=1e-3,
    epochs=5,
    steps_per_epoch=len(dl_train),
)
scheduler_callback = rnnm_learner.EveryBatchSchedulerCallback(scheduler)
learner.update_callback(scheduler_callback)

In [ ]:
learner.fit(dl_train_overfit, n_epochs=5)

In [ ]:
rnnm_learner.plot(callbacks)

In [ ]:
def summarize(learner: rnnm_learner.Learner, dl: DataLoader):
    y_prob = learner.predict(dl)
    assert isinstance(y_prob, torch.Tensor)
    ys_pred = y_prob.argmax(dim=1).numpy()
    ys_true = np.array([v.label.argmax(axis=1) for v in dl]).ravel()

    accuracy = metrics.accuracy_score(ys_true, ys_pred)
    error_rate = 1 - accuracy
    print(f"* Accuracy: {accuracy:.2%}")
    print(f"* Error rate: {error_rate:.2%}")

    metrics.ConfusionMatrixDisplay.from_predictions(ys_true, ys_pred, labels=range(10))
    plt.axis("off")
    plt.show()


summarize(learner, dl_train_overfit)

In [ ]:
summarize(learner, dl_valid)

In [ ]:
if DO_OVERFITTING_ONLY:
    raise SystemExit("Skipping training beyond overfitting.")

## Getting ~95% accuracy on 10k digits

In [ ]:
model = conv_lecun1990.Model2(lecun_init=True, lecun_act=True, dtype=dtype)

In [ ]:
n_epochs = 10
lr = 1e-3
# optimizer = optim.SGD(model.parameters(), lr=lr)  # , momentum=1e-3
optimizer = optim.Adam(model.parameters(), lr=lr)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer=optimizer,
    max_lr=lr,
    epochs=n_epochs,
    steps_per_epoch=len(dl_train),
)
loss = rnnm_losses.MSELossMNIST1HotLabel()
save_dir = Path("../../models").absolute().resolve()
save_dir.mkdir(exist_ok=True)


callbacks = rnnm_learner.get_default_train_callbacks(
    activation_callback_args=rnnm_learner.TelemetryArgs(100, 4, (r".*act.*",)),
    gradient_callback_args=rnnm_learner.TelemetryArgs(100, 4, (r"conv\d$", r"lin\d")),
    parameter_callback_args=rnnm_learner.TelemetryArgs(100, 4, (r"conv\d$", r"lin\d")),
    scheduler=scheduler,
)
lr_find_callback = rnnm_learner.LRFinderCallback(1e-5, 10, 100)

learner = rnnm_learner.Learner(
    model, optimizer, loss, callbacks=callbacks, save_dir=save_dir
)

In [ ]:
learner.find_learning_rate(dl_train, n_epochs=2, lr_find_callback=lr_find_callback)

In [ ]:
lr_find_callback.plot(log_xscale=True)

In [ ]:
learner.fit(dl_train, n_epochs=n_epochs, dataloader_valid=dl_valid)

In [ ]:
rnnm_learner.plot(callbacks)

computing test set performance

In [ ]:
summarize(learner, dl_valid)

In [ ]:
summarize(learner, dl_test)

* Accuracy: 95.15%
* Error rate: 4.85%